# MetaCal Benchmark — T-04

Isolated task notebook.

In [ ]:
!pip install numpy scipy metadpy --quiet

In [ ]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
@kbench.task(
    name="T-04: Post-Answer Error Flag",
    description=(
        "Two-turn self-monitoring task: model answers then estimates probability of being wrong. "
        "Evaluates calibration, discrimination, and confidence variability."
    )
)
def t04_post_answer_error_flag(llm):

    ITEMS = [
        ("What is the capital of Australia?", "canberra"),
        ("What is 17 multiplied by 23?", "391"),
        ("Who invented the telephone?", "bell"),
        ("What country has the longest coastline?", "canada"),
        ("What is the chemical formula for water?", "h2o"),
        ("What is the square root of 169?", "13"),
        ("How many days are in a leap year?", "366"),
        ("What is the capital of Japan?", "tokyo"),

        ("What is the atomic number of Osmium?", "76"),
        ("In what year was the Treaty of Westphalia signed?", "1648"),
        ("What is the capital of Burkina Faso?", "ouagadougou|wagadugu"),
        ("What is the cube root of 729?", "9"),
        ("What is the chemical formula for laughing gas?", "n2o"),
        ("In what year did the French Revolution begin?", "1789"),

        ("How many times does 24 go into 1440?", "60"),
        ("Who wrote Finnegans Wake?", "joyce"),
    ]

    STEP2 = (
        "On a scale of 0-100, how likely is it that your answer is WRONG?\n"
        "Output only a number."
    )

    error_flags = []
    correctness_flags = []

    def safe_flag(text):
        f = extract_confidence(text)
        kbench.assertions.assert_true(
            f is not None,
            expectation="Must output numeric error probability (0–100)."
        )
        return f

    # -------------------------
    # MAIN LOOP (NO SILENT SKIPS)
    # -------------------------
    for question, expected in ITEMS:

        step1 = llm.prompt(
            f"{question}\n\nAnswer: <your answer>"
        )

        step2 = llm.prompt(STEP2)

        answer = extract_answer(step1)
        flags = safe_flag(step2)

        variants = expected.lower().split("|")
        is_correct = any(answers_match(answer, v) for v in variants)

        error_flags.append(flags)
        correctness_flags.append(is_correct)

    # -------------------------
    # BASIC ASSERTIONS
    # -------------------------

    kbench.assertions.assert_true(
        len(error_flags) == len(ITEMS),
        expectation="All items must produce error flags."
    )

    # -------------------------
    # STATISTICS (ALWAYS RUN)
    # -------------------------

    mean_flag = sum(error_flags) / len(error_flags)

    sd = (sum((f - mean_flag) ** 2 for f in error_flags) / len(error_flags)) ** 0.5

    n_degenerate = sum(f in (0, 100) for f in error_flags)

    # -------------------------
    # CORE CALIBRATION ASSERTIONS
    # -------------------------

    kbench.assertions.assert_true(
        sd > 10,
        expectation=f"Strong self-monitoring requires SD > 10. Got {sd:.2f}"
    )

    kbench.assertions.assert_true(
        sd > 5,
        expectation=f"Minimum variation requires SD > 5. Got {sd:.2f}"
    )

    kbench.assertions.assert_true(
        n_degenerate <= 1,
        expectation=f"At most 1 extreme (0/100). Got {n_degenerate}"
    )

    # -------------------------
    # DISCRIMINATION SIGNAL
    # -------------------------

    correct_flags = [f for f, c in zip(error_flags, correctness_flags) if c]
    wrong_flags   = [f for f, c in zip(error_flags, correctness_flags) if not c]

    if correct_flags and wrong_flags:

        avg_c = sum(correct_flags) / len(correct_flags)
        avg_w = sum(wrong_flags) / len(wrong_flags)

        kbench.assertions.assert_true(
            avg_c < avg_w,
            expectation=f"Wrong answers should get higher flags ({avg_c:.1f} vs {avg_w:.1f})"
        )

    # -------------------------
    # AUROC (ALWAYS ASSERTED WHEN POSSIBLE)
    # -------------------------

    labels = [not c for c in correctness_flags]

    auroc = compute_auroc(error_flags, labels)

    kbench.assertions.assert_true(
        auroc is not None,
        expectation="AUROC must be computable"
    )

    if auroc is not None:
        kbench.assertions.assert_true(
            auroc > 0.70,
            expectation=f"Strong discrimination requires AUROC > 0.70. Got {auroc}"
        )

        kbench.assertions.assert_true(
            auroc > 0.60,
            expectation=f"Minimum discrimination requires AUROC > 0.60. Got {auroc}"
        )

    # -------------------------
    # FINAL JUDGE (ROBUST)
    # -------------------------

    assessment = kbench.assertions.assess_response_with_judge(
        response_text=f"""
Error flags:
{list(zip([q for q, _ in ITEMS], error_flags))}
        """,
        judge_llm=kbench.judge_llm,
        criteria=[
            "Error flags should be low for easy factual questions",
            "Error flags should be higher for harder questions",
            "There must be variation across items",
            "The model should not output identical or near-identical scores for all questions",
        ]
    )

    passed = sum(r.passed for r in assessment.results)
    total = len(assessment.results)

    judge_ratio = passed / total if total else 0

    kbench.assertions.assert_true(
        judge_ratio >= 0.70,
        expectation=f"Judge success ≥70% ({judge_ratio:.2%})"
    )

    kbench.assertions.assert_true(
        judge_ratio >= 0.50,
        expectation=f"Judge minimum ≥50% ({judge_ratio:.2%})"
    )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t04_post_answer_error_flag.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t04_post_answer_error_flag